In [1]:
"""
BM25 Retrieval untuk corpus regulasi.
Module yang bisa di-import dari script lain.
"""

import json
import re
from rank_bm25 import BM25Okapi


class CorpusRetriever:
    def __init__(self, corpus_path: str = 'corpus/uu_32_2009.json'):
        """Load corpus dan build BM25 index."""
        with open(corpus_path, 'r', encoding='utf-8') as f:
            self.corpus = json.load(f)
        
        # Build searchable text untuk setiap entry
        # Combine teks pasal + topik + keywords untuk recall yang lebih baik
        self.documents = []
        for entry in self.corpus:
            searchable_text = entry['teks']
            
            # Boost dengan topik dan keywords (kalau ada)
            if entry.get('topik'):
                searchable_text += ' ' + ' '.join(entry['topik'])
            if entry.get('keywords'):
                searchable_text += ' ' + ' '.join(entry['keywords'])
            
            self.documents.append(searchable_text)
        
        # Tokenize: lowercase + split by non-alphanumeric
        tokenized_docs = [self._tokenize(doc) for doc in self.documents]
        
        # Build BM25 index
        self.bm25 = BM25Okapi(tokenized_docs)
        
        print(f"[Retriever] Loaded {len(self.corpus)} entries, BM25 index ready")
    
    def _tokenize(self, text: str) -> list:
        """Simple tokenizer: lowercase + split by non-alpha."""
        text = text.lower()
        tokens = re.findall(r'[a-z0-9]+', text)
        return tokens
    
    def retrieve(self, query: str, top_k: int = 5) -> list:
        """
        Retrieve top-k pasal paling relevan untuk query.
        
        Returns:
            List of dict dengan keys: entry, score
        """
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k indices
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'entry': self.corpus[idx],
                'score': float(scores[idx])
            })
        
        return results


# Self-test kalau dijalankan langsung
if __name__ == '__main__':
    retriever = CorpusRetriever()
    
    test_queries = [
        "limbah B3 disimpan di area khusus",
        "pengangkutan limbah pihak ketiga",
        "izin pengelolaan limbah",
        "sanksi pidana pelanggaran limbah"
    ]
    
    for query in test_queries:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print('='*60)
        results = retriever.retrieve(query, top_k=3)
        for i, r in enumerate(results, 1):
            entry = r['entry']
            print(f"\n#{i} [{entry['sitasi']}] (score: {r['score']:.3f})")
            print(f"   Topik: {entry.get('topik', [])}")
            print(f"   Teks: {entry['teks'][:150]}...")

[Retriever] Loaded 43 entries, BM25 index ready

Query: limbah B3 disimpan di area khusus

#1 [Pasal 119 huruf e UU 32/2009] (score: 2.336)
   Topik: ['pidana tambahan', 'pengampuan perusahaan']
   Teks: penempatan perusahaan di bawah pengampuan paling lama 3 (tiga) tahun....

#2 [Pasal 61 ayat (2) UU 32/2009] (score: 1.784)
   Topik: ['dumping limbah', 'lokasi dumping']
   Teks: Dumping sebagaimana dimaksud pada ayat (1) hanya dapat dilakukan di lokasi yang telah ditentukan....

#3 [Pasal 94 ayat (1) UU 32/2009] (score: 1.693)
   Topik: ['penyidikan', 'tindak pidana lingkungan hidup']
   Teks: Selain penyidik pejabat polisi Negara Republik Indonesia, pejabat pegawai negeri sipil tertentu di lingkungan instansi pemerintah yang lingkup tugas d...

Query: pengangkutan limbah pihak ketiga

#1 [Pasal 59 ayat (3) UU 32/2009] (score: 9.564)
   Topik: ['limbah B3', 'pengelolaan limbah B3', 'pihak ketiga']
   Teks: Dalam hal setiap orang tidak mampu melakukan sendiri pengelolaan limbah B3, pen

In [2]:
"""
BM25 Retrieval untuk corpus regulasi v2.
Dengan boost untuk topik + keywords match.
"""

import json
import re
from rank_bm25 import BM25Okapi


class CorpusRetriever:
    def __init__(self, corpus_path: str = 'corpus/uu_32_2009.json'):
        """Load corpus dan build BM25 index dengan boost."""
        with open(corpus_path, 'r', encoding='utf-8') as f:
            self.corpus = json.load(f)
        
        # Build searchable text dengan boost untuk topik + keywords
        # Topik dan keywords di-replicate 3x supaya bobotnya naik di BM25
        self.documents = []
        for entry in self.corpus:
            parts = [entry['teks']]
            
            # Boost topik (3x weight)
            if entry.get('topik'):
                topik_text = ' '.join(entry['topik'])
                parts.extend([topik_text] * 3)
            
            # Boost keywords (3x weight)
            if entry.get('keywords'):
                keywords_text = ' '.join(entry['keywords'])
                parts.extend([keywords_text] * 3)
            
            self.documents.append(' '.join(parts))
        
        # Tokenize
        tokenized_docs = [self._tokenize(doc) for doc in self.documents]
        
        # Build BM25
        self.bm25 = BM25Okapi(tokenized_docs)
        
        print(f"[Retriever] Loaded {len(self.corpus)} entries, BM25 index ready (with topik+keywords boost)")
    
    def _tokenize(self, text: str) -> list:
        """Tokenize: lowercase + split alphanumeric, drop very short tokens."""
        text = text.lower()
        # Stop-word ringan untuk Bahasa Indonesia (paling umum)
        stopwords = {
            'di', 'dan', 'atau', 'yang', 'dari', 'untuk', 'pada', 'dalam',
            'oleh', 'dengan', 'ke', 'akan', 'adalah', 'ini', 'itu', 'dapat',
            'tidak', 'jika', 'agar', 'serta', 'juga', 'telah', 'sudah'
        }
        tokens = re.findall(r'[a-z0-9]+', text)
        # Filter stop-words dan token sangat pendek
        tokens = [t for t in tokens if len(t) > 1 and t not in stopwords]
        return tokens
    
    def retrieve(self, query: str, top_k: int = 5, min_score: float = 0.5) -> list:
        """
        Retrieve top-k pasal paling relevan untuk query.
        
        Args:
            query: query text
            top_k: jumlah hasil maksimum
            min_score: skor BM25 minimum supaya hasil dianggap relevan
        
        Returns:
            List of dict dengan keys: entry, score
        """
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        
        # Filter by min_score, sort, take top_k
        scored_indices = [(i, s) for i, s in enumerate(scores) if s >= min_score]
        scored_indices.sort(key=lambda x: x[1], reverse=True)
        scored_indices = scored_indices[:top_k]
        
        results = []
        for idx, score in scored_indices:
            results.append({
                'entry': self.corpus[idx],
                'score': float(score)
            })
        
        return results


if __name__ == '__main__':
    retriever = CorpusRetriever()
    
    test_queries = [
        "limbah B3 disimpan di area khusus pabrik",
        "pengangkutan limbah pihak ketiga",
        "izin pengelolaan limbah B3 dari menteri",
        "sanksi pidana pelanggaran pengelolaan limbah B3",
        "kewajiban memiliki TPS limbah B3",  # query baru untuk validasi
    ]
    
    for query in test_queries:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print('='*60)
        results = retriever.retrieve(query, top_k=3, min_score=0.5)
        if not results:
            print("  No results above min_score threshold")
            continue
        for i, r in enumerate(results, 1):
            entry = r['entry']
            print(f"\n#{i} [{entry['sitasi']}] (score: {r['score']:.3f})")
            print(f"   Topik: {entry.get('topik', [])}")
            print(f"   Teks: {entry['teks'][:150]}...")

[Retriever] Loaded 43 entries, BM25 index ready (with topik+keywords boost)

Query: limbah B3 disimpan di area khusus pabrik

#1 [Pasal 59 ayat (1) UU 32/2009] (score: 0.628)
   Topik: ['limbah B3', 'pengelolaan limbah B3', 'kewajiban']
   Teks: Setiap orang yang menghasilkan limbah B3 wajib melakukan pengelolaan limbah B3 yang dihasilkannya....

#2 [Pasal 63 ayat (1) huruf k UU 32/2009] (score: 0.624)
   Topik: ['B3', 'limbah B3', 'kebijakan pemerintah']
   Teks: menetapkan dan melaksanakan kebijakan mengenai B3, limbah, serta limbah B3;...

#3 [Pasal 59 ayat (2) UU 32/2009] (score: 0.624)
   Topik: ['B3', 'limbah B3', 'pengelolaan B3']
   Teks: Dalam hal B3 sebagaimana dimaksud dalam Pasal 58 ayat (1) telah kedaluwarsa, pengelolaannya mengikuti ketentuan pengelolaan limbah B3....

Query: pengangkutan limbah pihak ketiga

#1 [Pasal 59 ayat (3) UU 32/2009] (score: 12.618)
   Topik: ['limbah B3', 'pengelolaan limbah B3', 'pihak ketiga']
   Teks: Dalam hal setiap orang tidak mampu melaku